# 10m — Calibração do SVM multiclasse (custo de treinar 21 SVMs de pares e de prever)

**Pergunta:** quanto custa treinar e usar um SVM com kernel RBF **nas 7 classes** e no seu computador? Este notebook **mede** o custo (em vez de estimá-lo no escuro) para decidirmos o protocolo da busca de hiperparâmetros do SVM multiclasse. Ele **não escolhe hiperparâmetros**.

## Por que o multiclasse precisa de uma calibração própria
O SVM resolve 7 classes treinando **21 SVMs de pares** (um para cada par de classes; o scikit-learn faz isso por dentro, um par depois do outro). O custo do binário **não** serve de régua: o ajuste do multiclasse tem 239.671 linhas (o do binário tinha 81 mil), e cada um dos 21 pares só vê as linhas das suas duas classes. Quantos vetores de suporte o modelo todo terá, e quanto custa prever com eles, só se sabe medindo.

## O que é medido (e o que NÃO é repetido)
| Item | Como |
|---|---|
| **Dados** | As **mesmas views** do RF multiclasse (ajuste 239.671, validação 59.918, refit 299.589; sem os atributos de AUC alto). Os ausentes já estão tratados (mediana do treino + indicadores `_ausente`) |
| **Normalizador** | O **mesmo do binário**: `StandardScaler` só nas colunas numéricas, ajustado no próprio treino; indicadores `_ausente` em 0/1. **Não é comparado de novo** (a comparação do notebook 10 já mostrou que o SVM precisa de normalização, e os atributos são os mesmos). O notebook só **confere** que ele funciona |
| **Pesos** | `class_weight="balanced"`, como no RF multiclasse (o `LDAP` e o `NetBIOS` têm menos linhas) |
| **Custo × tamanho** | Treina o SVM multiclasse inteiro com 1 mil, 2 mil, 4 mil, 8 mil e 16 mil linhas **por classe** (como no 04m: "até N linhas de cada classe") e mede tempo, CPU, RAM, **vetores de suporte** (total e por classe) e o custo de prever (por par linha × vetor de suporte) |
| **Sensibilidade a C e gamma** | Treina as **9 configurações da grade** (C ∈ {1, 10, 100} × gamma ∈ {0,25×, 1×, 4×} do valor `scale`, a mesma do binário) num tamanho médio, para ver **quanto a grade custa a mais que a configuração base**. É só custo: não escolhe nada |
| **Estimativas** | Tempo da busca (CV de 5 dobras e só validação), do refit e da previsão do teste, para cada tamanho de amostra de busca, e uma **recomendação** de protocolo |
| **Cache do kernel e paralelismo** | **Não são medidos de novo**: já foram medidos neste computador nos notebooks 10 e 10b (cache de 200 MB por processo; prever com 2 processos dá ganho de 1,84×; treino com 1 processo). Entram como parâmetros (`CACHE_MB`, `GANHO_PREVISAO`, `GANHO_TREINO`) que você pode mudar |

## Proteções
Cada treino roda **num processo à parte, com limite de tempo** (`ORCAMENTO_FIT_S`, padrão 25 min): se passar, o processo é encerrado e o notebook segue. Um treino que, pela tendência, passaria do limite **nem é iniciado**. Se o SVM for lento, o notebook para de aumentar o tamanho e **extrapola** (marcando o que é extrapolação).

## O que o notebook grava
Na pasta `artefatos_modelos/svm_multiclasse_calibracao/svm_cal_<data>`: as tabelas (CSV), as estimativas, o gráfico (tempo e vetores de suporte × tamanho) e `calibracao_svm.json`. **Nenhum arquivo existente é alterado**, e o servidor só é lido.

## Antes de rodar
Os notebooks 02, 03m e 04m precisam ter terminado. **Feche os outros notebooks** e programas pesados, deixe o computador na tomada e sem suspensão. **Tempo estimado (sem medir): de ~40 a ~90 minutos**, conforme a velocidade do SVM nos seus dados (o tamanho de 16 mil por classe é o mais caro e pode ser pulado automaticamente pelo limite de tempo).


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

import gc, threading
def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 1. Parâmetros, dados e normalizador
Lê as mesmas views do RF multiclasse, confere o pré-processamento e o normalizador (o mesmo do binário) e mostra o computador.

In [6]:
# ---------- 1.1 Parâmetros, dados e normalizador ----------
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures

TAMANHOS_POR_CLASSE = tuple(globals().get("TAMANHOS_POR_CLASSE", (1_000, 2_000, 4_000, 8_000, 16_000)))     # linhas POR CLASSE (como no 04m: "até N linhas reais de cada classe")
C_VALORES = tuple(globals().get("C_VALORES", (1.0, 10.0, 100.0))); GAMMA_FATORES = tuple(globals().get("GAMMA_FATORES", (0.25, 1.0, 4.0)))        # a grade do binário (gamma = múltiplos do valor "scale")
CACHE_MB = globals().get("CACHE_MB", 200)                                   # cache do kernel de cada SVM (medido nos notebooks 10 e 10b)
ORCAMENTO_FIT_S = globals().get("ORCAMENTO_FIT_S", 1500)                    # nenhum treino pode passar disto (o processo é encerrado)
ORCAMENTO_SENS_S = globals().get("ORCAMENTO_SENS_S", 120)                   # a sensibilidade usa o maior tamanho cujo treino base levou até isto
LIMITE_BUSCA_H = globals().get("LIMITE_BUSCA_H", 6.0); LIMITE_TESTE_H = globals().get("LIMITE_TESTE_H", 1.5)           # para a recomendação (em horas)
GANHO_PREVISAO = globals().get("GANHO_PREVISAO", 1.84)                      # prever com 2 processos (medido no notebook 10)
GANHO_TREINO = globals().get("GANHO_TREINO", 1.0)                           # treino com 1 processo (a RAM livre limita o paralelismo de treino)
SEMENTE = globals().get("SEMENTE", 42); N_VAL_CAL = globals().get("N_VAL_CAL", None)      # None = a validação inteira
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PASTA = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_multiclasse_calibracao" / f"svm_cal_{RUN_ID}"; PASTA.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT, VIEW_TESTE = "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao", "prep_vw_bal_multi_refit", "prep_vw_rf_multi_teste_sem_sobrep"
AUD_TAB, TAB_CLASSES = "prep_views_rf_multi", "prep_classes_rf_multi"
faltam = [t for t in (V_AJUSTE, V_VALID, V_REFIT, VIEW_TESTE, AUD_TAB, TAB_CLASSES) if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03m e 04m: faltam {faltam}"
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo
assert CM == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
NUM, IND = list(range(len(prep.features))), list(range(len(prep.features), len(CM)))           # as colunas numéricas vêm primeiro; os indicadores _ausente, no fim
cl = consultar_num(f"SELECT codigo, classe FROM {TAB_CLASSES} ORDER BY codigo"); CLASSES = cl.classe.tolist(); K = len(CLASSES); assert cl.codigo.tolist() == list(range(K))

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
X_aj, y_aj, X_va, y_va = aj[CM].to_numpy(float), aj["y"].to_numpy(int), va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
N_AJ, N_VA, N_RF, N_TESTE = len(X_aj), len(X_va), n_linhas(V_REFIT), n_linhas(VIEW_TESTE); CONT_AJ = np.bincount(y_aj, minlength=K); assert (CONT_AJ > 0).all()

def novo_pipeline(C=1.0, gamma="scale", cache=None):
    # Pipeline do SVM: normaliza (média e desvio do próprio treino) só as colunas numéricas; os indicadores _ausente ficam 0/1; depois o SVM RBF multiclasse (21 pares), com class_weight="balanced" como no RF.
    pre = ColumnTransformer([("num", StandardScaler(), NUM)] + ([("ind", "passthrough", IND)] if IND else []))
    return Pipeline([("escala", pre), ("svm", SVC(kernel="rbf", C=C, gamma=gamma, cache_size=cache or CACHE_MB, class_weight="balanced"))])

def amostra(m, semente=SEMENTE):
    # até m linhas de CADA classe do ajuste (todas, se a classe tiver menos); None = o ajuste inteiro
    if m is None: return np.arange(N_AJ)
    r = np.random.default_rng(semente); return np.sort(np.concatenate([r.choice(np.where(y_aj == k)[0], size=min(m, int(CONT_AJ[k])), replace=False) for k in range(K)]))
def amostra_val(n):
    if n is None or n >= N_VA: return np.arange(N_VA)
    r = np.random.default_rng(SEMENTE); return np.sort(np.concatenate([r.choice(np.where(y_va == k)[0], size=max(1, int(round(n * (y_va == k).mean()))), replace=False) for k in range(K)]))
IV = amostra_val(N_VAL_CAL); X_vc, y_vc = X_va[IV], y_va[IV]

print(f"Pasta da calibração: {PASTA}")
print(f"{K} classes: {CLASSES} | ajuste: {N_AJ:,} linhas ({', '.join(f'{c} {int(n):,}' for c, n in zip(CLASSES, CONT_AJ))}) | validação: {N_VA:,} (usadas: {len(IV):,}) | refit: {N_RF:,} | teste: {N_TESTE:,}")
print(f"Atributos: {len(CM)} = {len(NUM)} numéricos (padronizados) + {len(IND)} indicadores _ausente (ficam 0/1)")
print(f"Computador: {os.cpu_count()} núcleos lógicos" + (f" ({psutil.cpu_count(logical=False)} físicos)" if psutil else "") + (f", {psutil.virtual_memory().total / 2 ** 30:.1f} GiB de RAM, {psutil.virtual_memory().available / 2 ** 30:.1f} GiB livres agora" if psutil else ""))
# conferência do normalizador: ajustado só no treino, numéricos com média 0 e desvio 1, indicadores intactos
_i = amostra(min(TAMANHOS_POR_CLASSE)); _p = novo_pipeline().fit(X_aj[_i], y_aj[_i]); _Xt = _p.named_steps["escala"].transform(X_aj[_i]); _sd = _Xt[:, NUM].std(0)
assert np.allclose(_Xt[:, NUM].mean(0), 0, atol=1e-9) and bool(((np.abs(_sd - 1) < 1e-6) | (_sd == 0)).all()), "o normalizador não deixou as colunas numéricas com média 0 e desvio 1"
assert (not IND) or np.array_equal(_Xt[:, IND], X_aj[_i][:, IND]), "os indicadores _ausente foram alterados pelo normalizador"
print("✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)")
T_SEC = {}


Pasta da calibração: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_calibracao\svm_cal_20261106_233729
7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag'] | ajuste: 239,671 linhas (BENIGN 40,000, LDAP 24,079, MSSQL 40,000, NetBIOS 15,592, Syn 40,000, UDP 40,000, UDPLag 40,000) | validação: 59,918 (usadas: 59,918) | refit: 299,589 | teste: 7,216,487
Atributos: 44 = 42 numéricos (padronizados) + 2 indicadores _ausente (ficam 0/1)
Computador: 12 núcleos lógicos (10 físicos), 7.7 GiB de RAM, 0.6 GiB livres agora
✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)


### Ferramentas da calibração
`_fit_worker_mc` treina o SVM multiclasse (21 pares) e mede treino e previsão; `executar_com_limite` roda cada treino **num processo à parte, com limite de tempo**; `ajustar_potencia` ajusta a tendência (tempo ou vetores de suporte × tamanho).

In [7]:
def _fit_worker_mc(X, y, Xv, yv, C, gamma, devolver_modelo=False):
    # Treina o SVM multiclasse (21 pares) e mede treino e previsão na validação. O escore é a decisão "um-contra-resto" do scikit-learn (decision_function, n x K); a classe é o argmax
    # (equivale a predict com desempate por confiança). Roda num processo à parte (executar_com_limite).
    pipe = novo_pipeline(C, gamma)
    with Medidor() as m: pipe.fit(X, y)
    svm, pre = pipe.named_steps["svm"], pipe.named_steps["escala"]
    t0 = time.perf_counter(); Xt = pre.transform(Xv); t_tr = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = svm.decision_function(Xt); t_sv = time.perf_counter() - t0
    nsv = int(svm.n_support_.sum()); K_ = sc.shape[1]; pred = sc.argmax(axis=1)
    cm = np.bincount(yv * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(float); t = cm.sum(1); p = cm.sum(0); d = np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    auc = [roc_auc_score(yv == k, sc[:, k]) for k in range(K_) if 0 < (yv == k).sum() < len(yv)]
    r = {"n": int(len(y)), "C": float(C), "gamma": float(svm._gamma), "tempo_treino_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib,
         "n_sv": nsv, "frac_sv": nsv / len(y), "n_sv_por_classe": [int(v) for v in svm.n_support_], "tempo_transformacao_val_s": t_tr, "tempo_svm_val_s": t_sv, "ns_por_par": t_sv / (len(Xv) * max(nsv, 1)) * 1e9,
         "roc_auc_ovr_macro": float(np.mean(auc)), "acuracia_balanceada": float(np.nanmean(rec)), "f1_macro": float(f1.mean()), "acuracia": float(d.sum() / cm.sum())}
    return (r, pipe if devolver_modelo else None)

def executar_com_limite(fn, args, limite_s, barra=None, rotulo=""):
    # Um processo NOVO e independente por chamada (não compartilha nada com o Parallel do joblib); se passar do limite, é encerrado.
    # Espera em fatias de 0,5 s: o botão "interromper" do Jupyter funciona (também no Windows) e a barra mostra o tempo decorrido.
    ex = LokyPool(max_workers=1); fut = ex.submit(fn, *args); t0 = time.perf_counter()
    try:
        while True:
            decorrido = time.perf_counter() - t0
            if decorrido >= limite_s: ex.shutdown(wait=True, kill_workers=True); return None
            if barra is not None: barra._extra = f"{rotulo} | em andamento há {fmt_tempo(decorrido)} (limite {fmt_tempo(limite_s)})"; barra._mostrar()
            try: res = fut.result(timeout=min(0.5, limite_s - decorrido))
            except (concurrent.futures.TimeoutError, TimeoutError): continue
            ex.shutdown(wait=True); return res
    except BaseException:
        ex.shutdown(wait=True, kill_workers=True); raise

def ajustar_potencia(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); ok_ = (x > 0) & (y > 0); b, loga = np.polyfit(np.log(x[ok_]), np.log(y[ok_]), 1)
    return (lambda n: float(np.exp(loga) * float(n) ** b)), float(b)
print("Ferramentas da calibração prontas")


Ferramentas da calibração prontas


## 2. Custo × tamanho
Treina o SVM multiclasse inteiro com 1 mil, 2 mil, 4 mil, 8 mil e 16 mil linhas **por classe** (C = 1, gamma = `scale`) e mede tempo, CPU, RAM, vetores de suporte e o custo de prever. Para quando o próximo treino passaria do limite de tempo; o resto vira **extrapolação** (a tendência em lei de potência).

In [8]:
# ---------- 3.1 Custo × tamanho ----------
t_ini = time.perf_counter(); res_tam, MODELO = [], None; barra = Barra(len(TAMANHOS_POR_CLASSE), "Custo × tamanho")
for m in TAMANHOS_POR_CLASSE:
    idx = amostra(m); n_real = len(idx)
    if len(res_tam) >= 2:
        (n1, t1), (n2, t2) = [(r["n"], r["tempo_treino_s"]) for r in res_tam[-2:]]; b = np.log(t2 / t1) / np.log(n2 / n1) if (t1 > 0 and t2 > 0 and n2 > n1) else 2.0; prev = t2 * (n_real / n2) ** max(b, 1.0)
        if prev > ORCAMENTO_FIT_S:
            print(f"  ⚠ o treino com {m:,} linhas por classe ({n_real:,} no total) é previsto em ~{fmt_tempo(prev)}, acima do limite ({fmt_tempo(ORCAMENTO_FIT_S)}): não será rodado (as estimativas usarão a tendência dos tamanhos menores)"); barra.avancar(1, f"{m:,} por classe: pulado"); break
    barra.avancar(0, f"treinando com {m:,} linhas por classe ({n_real:,} no total; limite {fmt_tempo(ORCAMENTO_FIT_S)})...")
    out = executar_com_limite(_fit_worker_mc, (X_aj[idx], y_aj[idx], X_vc, y_vc, 1.0, "scale", True), ORCAMENTO_FIT_S, barra, f"treinando com {n_real:,} linhas")
    if out is None: print(f"  ⚠ o treino com {m:,} linhas por classe passou de {fmt_tempo(ORCAMENTO_FIT_S)} e foi encerrado"); barra.avancar(1, "passou do limite"); break
    r, pipe = out; r["por_classe"] = int(m); res_tam.append(r); MODELO = pipe
    barra.avancar(1, f"{m:,} por classe: treino {fmt_tempo(r['tempo_treino_s'])}, {r['n_sv']:,} vetores de suporte ({100 * r['frac_sv']:.0f}%), ROC-AUC {r['roc_auc_ovr_macro']:.5f}")
barra.fechar(); assert res_tam, "nenhum treino foi concluído dentro do limite de tempo: aumente ORCAMENTO_FIT_S ou diminua TAMANHOS_POR_CLASSE"
tab_tam = pd.DataFrame(res_tam); print(f"\n== Custo × tamanho (validação com {len(IV):,} linhas; C = 1, gamma = scale, class_weight = balanced) ==")
display(tab_tam[["por_classe", "n", "tempo_treino_s", "cpu_s", "pico_ram_mib", "n_sv", "frac_sv", "tempo_svm_val_s", "ns_por_par", "roc_auc_ovr_macro", "acuracia_balanceada", "f1_macro"]].round(4))
N_MAX_MEDIDO = int(tab_tam["n"].max())
if len(res_tam) >= 2: f_t, b_t = ajustar_potencia(tab_tam["n"], tab_tam["tempo_treino_s"]); f_sv, b_sv = ajustar_potencia(tab_tam["n"], tab_tam["n_sv"]); SUPOSTO = False
else:
    r0 = res_tam[0]; b_t, b_sv = 2.0, 1.0; f_t = lambda n, r0=r0: r0["tempo_treino_s"] * (n / r0["n"]) ** 2.0; f_sv = lambda n, r0=r0: r0["n_sv"] * (n / r0["n"]); SUPOSTO = True
NS_PAR = float(tab_tam["ns_por_par"].iloc[-min(2, len(tab_tam)):].mean())
print(f"\nO tempo de treino cresce como n^{b_t:.2f} e os vetores de suporte como n^{b_sv:.2f} (n = linhas totais de treino)" + ("  (⚠ só um tamanho medido: expoentes SUPOSTOS 2 e 1)" if SUPOSTO else f"  (ajuste com {len(res_tam)} tamanhos)") + f" | custo de previsão: ~{NS_PAR:.0f} ns por par (linha × vetor de suporte)")
print("Vetores de suporte por classe (maior tamanho medido): " + ", ".join(f"{c} {v:,}" for c, v in zip(CLASSES, res_tam[-1]["n_sv_por_classe"])))
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4)); xs = np.linspace(tab_tam["n"].min(), max(N_RF, N_MAX_MEDIDO), 50)
    ax[0].loglog(tab_tam["n"], tab_tam["tempo_treino_s"], "o", label="medido"); ax[0].loglog(xs, [f_t(v) for v in xs], "--", label=f"tendência n^{b_t:.2f}"); ax[0].set_xlabel("linhas de treino (total)"); ax[0].set_ylabel("tempo de treino (s)"); ax[0].legend(); ax[0].grid(alpha=.3, which="both")
    ax[1].loglog(tab_tam["n"], tab_tam["n_sv"], "o", label="medido"); ax[1].loglog(xs, [f_sv(v) for v in xs], "--", label=f"tendência n^{b_sv:.2f}"); ax[1].set_xlabel("linhas de treino (total)"); ax[1].set_ylabel("vetores de suporte (total)"); ax[1].legend(); ax[1].grid(alpha=.3, which="both")
    fig.suptitle("SVM multiclasse (21 pares): custo × tamanho"); fig.tight_layout(); fig.savefig(PASTA / "custo_por_tamanho.png", dpi=130); plt.close(fig)
except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
T_SEC["3.1_tamanho"] = time.perf_counter() - t_ini


Custo × tamanho: concluído em 23min 24s

== Custo × tamanho (validação com 59,918 linhas; C = 1, gamma = scale, class_weight = balanced) ==


,por_classe,n,tempo_treino_s,cpu_s,pico_ram_mib,n_sv,frac_sv,tempo_svm_val_s,ns_por_par,roc_auc_ovr_macro,acuracia_balanceada,f1_macro
0,1000,7000,0.9622,0.9688,189.6250,2993,0.4276,42.1078,234.8003,0.9732,0.8517,0.8542
1,2000,14000,3.2356,3.2812,225.2852,5500,0.3929,58.6330,177.9189,0.9745,0.8553,0.8581
2,4000,28000,15.6102,15.5781,371.5195,9913,0.3540,110.6380,186.2696,0.9757,0.8575,0.8597
3,8000,56000,105.5156,104.6562,437.8477,18218,0.3253,183.3949,168.0077,0.9763,0.8593,0.8635
4,16000,111592,443.3868,442.2812,505.5508,34923,0.3130,423.9888,202.6215,0.9761,0.8618,0.8633



O tempo de treino cresce como n^2.27 e os vetores de suporte como n^0.88 (n = linhas totais de treino)  (ajuste com 5 tamanhos) | custo de previsão: ~185 ns por par (linha × vetor de suporte)
Vetores de suporte por classe (maior tamanho medido): BENIGN 1,240, LDAP 2,167, MSSQL 10,134, NetBIOS 9,715, Syn 1,947, UDP 4,032, UDPLag 5,688


## 3. Sensibilidade a C e gamma
Treina as **9 configurações da grade** (a mesma do binário) num tamanho médio, só para saber **quanto a grade custa a mais que a configuração base**. Não escolhe nada: o ROC-AUC e a acurácia balanceada são informativos.

In [9]:
# ---------- 4.1 Sensibilidade a C e gamma (a grade inteira, num tamanho médio) ----------
t_ini = time.perf_counter(); cand = [r["por_classe"] for r in res_tam if r["tempo_treino_s"] <= ORCAMENTO_SENS_S]; M_SENS = max(cand) if cand else int(tab_tam["por_classe"].min())
base = next(r for r in res_tam if r["por_classe"] == M_SENS); gamma0 = base["gamma"]; idx_s = amostra(M_SENS); Xs, ys = X_aj[idx_s], y_aj[idx_s]
LIMITE_SENS_S = min(ORCAMENTO_FIT_S, max(180, 10 * base["tempo_treino_s"]))               # um treino 10× mais lento que o da base já é caro demais para a grade
print(f"Amostra da sensibilidade: {M_SENS:,} linhas por classe ({len(idx_s):,} no total) | gamma 'scale' = {gamma0:.5f} | base (C = 1, gamma = scale): treino {fmt_tempo(base['tempo_treino_s'])}, {base['n_sv']:,} vetores de suporte")
print(f"Limite por treino nesta seção: {fmt_tempo(LIMITE_SENS_S)} (10× o da base, no mínimo 3 min); quem passar é encerrado e marcado")
combos = [(C, f) for C in C_VALORES for f in GAMMA_FATORES if not (C == 1.0 and f == 1.0)]; linhas_sens = [{**base, "gamma_fator": 1.0, "concluiu": True}]; barra = Barra(len(combos), "Sensibilidade a C e gamma")
for C, f in combos:
    barra.avancar(0, f"C = {C:g}, gamma = {f:g}×scale...")
    out = executar_com_limite(_fit_worker_mc, (Xs, ys, X_vc, y_vc, C, gamma0 * f, False), LIMITE_SENS_S, barra, f"C = {C:g}, gamma = {f:g}×scale")
    if out is None: linhas_sens.append({"n": len(idx_s), "C": C, "gamma": gamma0 * f, "gamma_fator": f, "tempo_treino_s": float("nan"), "concluiu": False}); barra.avancar(1, f"C = {C:g}, gamma = {f:g}×scale: PASSOU DO LIMITE"); continue
    r = out[0]; linhas_sens.append({**r, "gamma_fator": f, "concluiu": True}); barra.avancar(1, f"C = {C:g}, gamma = {f:g}×scale: {fmt_tempo(r['tempo_treino_s'])}, {r['n_sv']:,} vetores de suporte")
barra.fechar(); tab_sens = pd.DataFrame(linhas_sens).sort_values(["C", "gamma_fator"]).reset_index(drop=True)
print("\n== Sensibilidade a C e gamma (informativa: só mede o custo da grade) =="); display(tab_sens[["C", "gamma_fator", "gamma", "tempo_treino_s", "n_sv", "frac_sv", "ns_por_par", "roc_auc_ovr_macro", "acuracia_balanceada", "f1_macro", "concluiu"]].round(5))
ok_s = tab_sens[tab_sens.concluiu]; t_base = float(base["tempo_treino_s"]); FATOR_MEDIO_GRADE = float(ok_s["tempo_treino_s"].mean() / t_base); FATOR_MAX_GRADE = float(ok_s["tempo_treino_s"].max() / t_base)
RAZAO_SV_MEDIA = float(ok_s["n_sv"].mean() / base["n_sv"]); n_perdidos = int((~tab_sens.concluiu).sum()); canto = tab_sens[(tab_sens.C == max(C_VALORES)) & (tab_sens.gamma_fator == max(GAMMA_FATORES))].iloc[0]
FATOR_T_CANTO = float(canto["tempo_treino_s"] / t_base) if bool(canto["concluiu"]) else float("nan"); RAZAO_SV_CANTO = float(canto["n_sv"] / base["n_sv"]) if bool(canto["concluiu"]) else float("nan")
print(f"O tempo de treino de uma configuração da grade é, em média, {FATOR_MEDIO_GRADE:.2f}× o da base (máximo {FATOR_MAX_GRADE:.2f}×); os vetores de suporte, {RAZAO_SV_MEDIA:.2f}× os da base." + (f" ⚠ {n_perdidos} configuração(ões) passaram do limite e NÃO entram nas médias (o custo real da grade é maior)." if n_perdidos else ""))
if bool(canto["concluiu"]): print(f"O canto C = {max(C_VALORES):g}, gamma = {max(GAMMA_FATORES):g}× (o que o binário escolheu) custa {FATOR_T_CANTO:.2f}× o tempo e {RAZAO_SV_CANTO:.2f}× os vetores de suporte da base.")
T_SEC["4.1_sensibilidade"] = time.perf_counter() - t_ini


Amostra da sensibilidade: 8,000 linhas por classe (56,000 no total) | gamma 'scale' = 0.02378 | base (C = 1, gamma = scale): treino 1min 46s, 18,218 vetores de suporte
Limite por treino nesta seção: 17min 35s (10× o da base, no mínimo 3 min); quem passar é encerrado e marcado


Sensibilidade a C e gamma: concluído em 40min 06s

== Sensibilidade a C e gamma (informativa: só mede o custo da grade) ==


,C,gamma_fator,gamma,tempo_treino_s,n_sv,frac_sv,ns_por_par,roc_auc_ovr_macro,acuracia_balanceada,f1_macro,concluiu
0,1.0,0.25,0.00594,149.19726,20687,0.36941,184.95475,0.96926,0.85599,0.85780,True
1,1.0,1.00,0.02378,105.51563,18218,0.32532,168.00769,0.97630,0.85930,0.86354,True
2,1.0,4.00,0.09512,110.71009,17586,0.31404,172.55411,0.97796,0.86537,0.87041,True
3,10.0,0.25,0.00594,97.71123,17578,0.31389,170.53287,0.97265,0.85938,0.86240,True
4,10.0,1.00,0.02378,95.82879,16921,0.30216,172.85486,0.97607,0.86532,0.87054,True
5,10.0,4.00,0.09512,121.01044,16812,0.30021,162.83339,0.97503,0.86785,0.87282,True
6,100.0,0.25,0.00594,94.65223,16580,0.29607,168.62932,0.97291,0.86256,0.86854,True
7,100.0,1.00,0.02378,101.11005,16303,0.29113,175.04079,0.96997,0.86785,0.87285,True
8,100.0,4.00,0.09512,138.08300,16542,0.29539,211.52666,0.97387,0.86943,0.87419,True


O tempo de treino de uma configuração da grade é, em média, 1.07× o da base (máximo 1.41×); os vetores de suporte, 0.96× os da base.
O canto C = 100, gamma = 4× (o que o binário escolheu) custa 1.31× o tempo e 0.91× os vetores de suporte da base.


## 4. Estimativas e recomendação
Com as medidas, estima o tempo da **busca** (grade 3 × 3, CV de 5 dobras ou só validação) em amostras de vários tamanhos, do **refit** e da **previsão do teste**, e recomenda o maior tamanho de amostra que cabe nos limites. **São extrapolações**: a coluna `extrapolado` marca as que usam tamanhos maiores que o maior medido. Valem como ordem de grandeza; a primeira execução real da busca dirá o resto.

In [10]:
# ---------- 5.1 Estimativas e recomendação ----------
prev_s = lambda linhas, n_sv: linhas * n_sv * NS_PAR * 1e-9                                  # segundos para prever `linhas` com um modelo de `n_sv` vetores de suporte, 1 processo
n_total = lambda m: int(sum(min(m, int(c)) for c in CONT_AJ)) if m is not None else N_AJ
CAND = [m for m in (2_000, 4_000, 8_000, 16_000, int(CONT_AJ.max())) if m <= int(CONT_AJ.max())]; CAND = sorted(set(CAND)); linhas = []
for m in CAND:
    N = n_total(m); Nt, Nv = 0.8 * N, 0.2 * N                                                 # treino e avaliação de cada dobra da CV
    t_cv = FATOR_MEDIO_GRADE * (f_t(Nt) + prev_s(Nv, f_sv(Nt) * RAZAO_SV_MEDIA)); t_full = FATOR_MEDIO_GRADE * (f_t(N) + prev_s(N_VA, f_sv(N) * RAZAO_SV_MEDIA))
    cv5 = len(C_VALORES) * len(GAMMA_FATORES) * (5 * t_cv + t_full) / GANHO_TREINO; so_val = len(C_VALORES) * len(GAMMA_FATORES) * t_full / GANHO_TREINO
    linhas.append({"linhas por classe": m, "linhas totais": N, "extrapolado": N > N_MAX_MEDIDO * 1.05, "busca CV 5 dobras (h)": cv5 / 3600, "busca só validação (h)": so_val / 3600})
tab_prot = pd.DataFrame(linhas)
def custo_final(fator_t, razao_sv):
    ref = f_t(N_RF) * fator_t / GANHO_TREINO; sv = f_sv(N_RF) * razao_sv; tst = prev_s(N_TESTE, sv) / GANHO_PREVISAO
    return {"refit (h)": ref / 3600, "vetores de suporte (milhares)": sv / 1e3, "previsão do teste (h)": tst / 3600}
fin_base = custo_final(1.0, 1.0); fin_canto = custo_final(FATOR_T_CANTO, RAZAO_SV_CANTO) if np.isfinite(FATOR_T_CANTO) else None
tab_fin = pd.DataFrame({"como a configuração base (C = 1, gamma = scale)": fin_base, **({f"como o canto C = {max(C_VALORES):g}, gamma = {max(GAMMA_FATORES):g}×": fin_canto} if fin_canto else {})}).T
print(f"== Estimativas (tempo de treino ~ n^{b_t:.2f}; maior tamanho medido: {N_MAX_MEDIDO:,} linhas; custo de previsão {NS_PAR:.0f} ns por par; fator médio da grade {FATOR_MEDIO_GRADE:.2f}×; ganho do treino {GANHO_TREINO:.2f}× e da previsão {GANHO_PREVISAO:.2f}×) ==")
print("\n-- Busca de hiperparâmetros (9 configurações da grade) em amostras do ajuste, conforme o tamanho --"); display(tab_prot.round(2))
print(f"\n-- Modelo final: refit com as {N_RF:,} linhas do refit e previsão do teste ({N_TESTE:,} linhas); os vetores de suporte e o tempo dependem da configuração escolhida --"); display(tab_fin.round(2))
print(f"\n== Recomendação (limites: busca ≤ {LIMITE_BUSCA_H:.1f} h, previsão do teste ≤ {LIMITE_TESTE_H:.1f} h) ==")
cabe_cv = tab_prot[tab_prot["busca CV 5 dobras (h)"] <= LIMITE_BUSCA_H]; cabe_val = tab_prot[tab_prot["busca só validação (h)"] <= LIMITE_BUSCA_H]
if len(cabe_cv): m_rec = int(cabe_cv["linhas por classe"].max()); REC = "cv5"; print(f"✔ A busca com CV de 5 dobras (a mesma do RF, com as mesmas dobras) cabe com até {m_rec:,} linhas por classe (~{float(cabe_cv.iloc[-1]['busca CV 5 dobras (h)']):.1f} h): este é o tamanho recomendado para a amostra da busca.")
elif len(cabe_val): m_rec = int(cabe_val["linhas por classe"].max()); REC = "validacao"; print(f"⚠ A CV completa não cabe em nenhum tamanho. Só a validação separada cabe, com até {m_rec:,} linhas por classe: use-a na busca e a CV de 5 dobras só nas finalistas.")
else: m_rec = int(tab_prot["linhas por classe"].min()); REC = "amostra_minima"; print(f"✘ Nem a validação separada cabe no limite com {m_rec:,} linhas por classe: reduza a grade ou os tamanhos (TAMANHOS_POR_CLASSE) e rode de novo.")
for nome, f_ in (("base", fin_base), ("canto", fin_canto)):
    if f_: print(("✔" if f_["previsão do teste (h)"] <= LIMITE_TESTE_H else "⚠") + f" Previsão do teste com um modelo {nome}: ~{f_['previsão do teste (h)']:.1f} h" + ("" if f_["previsão do teste (h)"] <= LIMITE_TESTE_H else " (passa do limite; a previsão em lotes com checkpoint e 2 processos ainda assim é viável, mas longa)") + f" | refit ~{f_['refit (h)']:.1f} h")
print("\nObservação sobre a comparação justa: o RF multiclasse usou o ajuste inteiro (239.671 linhas). Se o SVM só puder usar uma amostra menor na busca, o RF também pode ser treinado nessa mesma amostra (custa segundos), para separar o efeito do tamanho do efeito do algoritmo.")
if any(r["extrapolado"] for r in linhas): print("(⚠ linhas marcadas 'extrapolado' usam tamanhos maiores que o maior medido: confirme com a primeira execução real.)")


== Estimativas (tempo de treino ~ n^2.27; maior tamanho medido: 111,592 linhas; custo de previsão 185 ns por par; fator médio da grade 1.07×; ganho do treino 1.00× e da previsão 1.84×) ==

-- Busca de hiperparâmetros (9 configurações da grade) em amostras do ajuste, conforme o tamanho --


,linhas por classe,linhas totais,extrapolado,busca CV 5 dobras (h),busca só validação (h)
0,2000,14000,False,0.23,0.17
1,4000,28000,False,0.60,0.34
2,8000,56000,False,1.90,0.77
3,16000,111592,False,7.11,2.13
4,40000,239671,True,34.63,8.50



-- Modelo final: refit com as 299,589 linhas do refit e previsão do teste (7,216,487 linhas); os vetores de suporte e o tempo dependem da configuração escolhida --


,refit (h),vetores de suporte (milhares),previsão do teste (h)
"como a configuração base (C = 1, gamma = scale)",1.14,81.67,16.49
"como o canto C = 100, gamma = 4×",1.49,74.16,14.97



== Recomendação (limites: busca ≤ 6.0 h, previsão do teste ≤ 1.5 h) ==
✔ A busca com CV de 5 dobras (a mesma do RF, com as mesmas dobras) cabe com até 8,000 linhas por classe (~1.9 h): este é o tamanho recomendado para a amostra da busca.
⚠ Previsão do teste com um modelo base: ~16.5 h (passa do limite; a previsão em lotes com checkpoint e 2 processos ainda assim é viável, mas longa) | refit ~1.1 h
⚠ Previsão do teste com um modelo canto: ~15.0 h (passa do limite; a previsão em lotes com checkpoint e 2 processos ainda assim é viável, mas longa) | refit ~1.5 h

Observação sobre a comparação justa: o RF multiclasse usou o ajuste inteiro (239.671 linhas). Se o SVM só puder usar uma amostra menor na busca, o RF também pode ser treinado nessa mesma amostra (custa segundos), para separar o efeito do tamanho do efeito do algoritmo.
(⚠ linhas marcadas 'extrapolado' usam tamanhos maiores que o maior medido: confirme com a primeira execução real.)


## 5. Gravação e conferências

In [11]:
# ---------- 6.1 Gravação e conferências ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
tab_tam.drop(columns=["n_sv_por_classe"]).to_csv(PASTA / "calibracao_tamanhos.csv", index=False); tab_sens.drop(columns=["n_sv_por_classe"], errors="ignore").to_csv(PASTA / "sensibilidade_grade.csv", index=False)
tab_prot.to_csv(PASTA / "estimativas_busca.csv", index=False); tab_fin.reset_index(names="modelo").to_csv(PASTA / "estimativas_modelo_final.csv", index=False)
pd.DataFrame({"classe": CLASSES, "vetores_de_suporte": res_tam[-1]["n_sv_por_classe"], "linhas_no_treino": [min(res_tam[-1]["por_classe"], int(c)) for c in CONT_AJ]}).to_csv(PASTA / "vetores_de_suporte_por_classe.csv", index=False)
salvar_json({"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "hardware": hw, "software": sw, "normalizador": "StandardScaler só nas colunas numéricas, ajustado no próprio treino; indicadores _ausente em 0/1 (o mesmo do binário)", "class_weight": "balanced",
             "parametros": {"tamanhos_por_classe": TAMANHOS_POR_CLASSE, "c_valores": C_VALORES, "gamma_fatores": GAMMA_FATORES, "cache_mb": CACHE_MB, "orcamento_fit_s": ORCAMENTO_FIT_S, "orcamento_sens_s": ORCAMENTO_SENS_S, "ganho_previsao": GANHO_PREVISAO, "ganho_treino": GANHO_TREINO},
             "medidas": {"maior_tamanho_medido": N_MAX_MEDIDO, "expoente_tempo": b_t, "expoente_vetores_suporte": b_sv, "ns_por_par": NS_PAR, "expoentes_supostos": SUPOSTO, "amostra_sensibilidade_por_classe": M_SENS, "gamma_scale": gamma0, "fator_medio_grade": FATOR_MEDIO_GRADE,
                         "fator_max_grade": FATOR_MAX_GRADE, "razao_sv_media": RAZAO_SV_MEDIA, "fator_tempo_canto": FATOR_T_CANTO, "razao_sv_canto": RAZAO_SV_CANTO, "configuracoes_que_passaram_do_limite": n_perdidos},
             "estimativas_busca": tab_prot.to_dict(orient="records"), "estimativas_modelo_final": tab_fin.to_dict(orient="index"), "recomendacao": {"protocolo": REC, "linhas_por_classe": m_rec}, "limites_h": {"busca": LIMITE_BUSCA_H, "teste": LIMITE_TESTE_H},
             "tempos_etapas_s": T_SEC, "observacao": "calibração de custo; não escolhe hiperparâmetros. ROC-AUC e acurácia da validação são informativos"}, PASTA / "calibracao_svm.json")
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Amostras por classe: cada classe tem min(m, linhas da classe) linhas, em todos os tamanhos medidos", all(np.array_equal(np.bincount(y_aj[amostra(r["por_classe"])], minlength=K), np.minimum(r["por_classe"], CONT_AJ)) for r in res_tam), f"({len(res_tam)} tamanhos)")
_i = amostra(min(TAMANHOS_POR_CLASSE)); _pp = novo_pipeline().fit(X_aj[_i], y_aj[_i]); _X3 = _pp.named_steps["escala"].transform(X_aj[_i]); _sd = _X3[:, NUM].std(0)
verifica("Normalizador sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos", np.allclose(_X3[:, NUM].mean(0), 0, atol=1e-9) and bool(((np.abs(_sd - 1) < 1e-6) | (_sd == 0)).all()) and ((not IND) or np.array_equal(_X3[:, IND], X_aj[_i][:, IND])))
verifica("O modelo multiclasse tem as K classes e vetores de suporte em todas elas", all(len(r["n_sv_por_classe"]) == K and min(r["n_sv_por_classe"]) > 0 for r in res_tam))
verifica("Vetores de suporte ≤ linhas de treino em todas as medidas", bool((tab_tam.n_sv <= tab_tam.n).all() and (ok_s.n_sv <= ok_s.n).all()))
verifica("ROC-AUC (um-contra-resto, macro) da validação entre 0 e 1 em todas as medidas", bool(tab_tam.roc_auc_ovr_macro.between(0, 1).all() and ok_s.roc_auc_ovr_macro.between(0, 1).all()))
verifica("A sensibilidade cobre a grade inteira (todas as combinações de C e gamma)", len(tab_sens) == len(C_VALORES) * len(GAMMA_FATORES) and set(zip(tab_sens.C, tab_sens.gamma_fator)) == {(c, f) for c in C_VALORES for f in GAMMA_FATORES})
verifica("Estimativas finitas e positivas", bool(np.isfinite(tab_prot.drop(columns=["extrapolado"]).to_numpy(float)).all() and (tab_prot[["busca CV 5 dobras (h)", "busca só validação (h)"]].to_numpy() > 0).all() and np.isfinite(tab_fin.to_numpy(float)).all() and (tab_fin.to_numpy(float) > 0).all()))
verifica("Arquivos gravados", all((PASTA / f).exists() for f in ("calibracao_tamanhos.csv", "sensibilidade_grade.csv", "estimativas_busca.csv", "estimativas_modelo_final.csv", "vetores_de_suporte_por_classe.csv", "calibracao_svm.json", "custo_por_tamanho.png")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print("\nPróximo passo: me mande o texto desta saída; com estes números defino o protocolo da busca do SVM multiclasse (notebook 11m).")


✔ Amostras por classe: cada classe tem min(m, linhas da classe) linhas, em todos os tamanhos medidos (5 tamanhos)
✔ Normalizador sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos 
✔ O modelo multiclasse tem as K classes e vetores de suporte em todas elas 
✔ Vetores de suporte ≤ linhas de treino em todas as medidas 
✔ ROC-AUC (um-contra-resto, macro) da validação entre 0 e 1 em todas as medidas 
✔ A sensibilidade cobre a grade inteira (todas as combinações de C e gamma) 
✔ Estimativas finitas e positivas 
✔ Arquivos gravados 

✔ 8/8 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_calibracao\svm_cal_20261106_233729

⏱ Tempo total do notebook (etapas medidas): 1h 03min

Próximo passo: me mande o texto desta saída; com estes números defino o protocolo da busca do SVM multiclasse (notebook 11m).
